In [5]:
import pickle 
import pandas as pd
import numpy as np
import json

seeds = [0,1,2,3,42]
cancers = ['ccRCC', 'Melanoma', 'NSCLC']

## 1. Singling Out

In [6]:
def load_results_SO(path, is_uni = True):
    with open(path, 'rb') as file:
        linkabilityresults = pickle.load(file)
    
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    if is_uni:
        columns = ['25%', '50%', '75%', '100%']
    else:
        columns = [2,3,5,7,10,20,50]
    ResultasDataset = {}
    for tool, result_as_tool in linkabilityresults.items():
        tool_res = {}
        for i, evaluator in enumerate(result_as_tool):    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[columns[i]] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools
    df.columns = columns
    return df

In [7]:
## Uni Risk
UniRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/SinglingOut/UniSO/Seed_{seed}/SOUni_Results.pkl'
        result_df = load_results_SO(data_path)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    UniRisk_Cancer[cancer] = ResultsLinkability

ModuleNotFoundError: No module named 'anonymeter'

In [ ]:
## Multi Risk
MultiRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/SinglingOut/MultiSO/Seed_{seed}/MultiSO_Results.pkl'
        result_df = load_results_SO(data_path, is_uni = False)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    MultiRisk_Cancer[cancer] = ResultsLinkability

## Linkability

In [ ]:
def load_results_linkability(path):
    with open(path, 'rb') as file:
        linkabilityresults = pickle.load(file)
    
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    columns = ['25%', '50%', '75%', '100%']
    ResultasDataset = {}
    for tool, result_as_tool in linkabilityresults.items():
        tool_res = {}
        for i, evaluator in enumerate(result_as_tool):
            # name_feature = feature[0]
            # evaluator = feature[1]
    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[columns[i]] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools
    df.columns = columns
    return df

In [ ]:
## Linkability Risk
LinkRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/Linkability/Seed_{seed}/LinkabilityResults.pkl'
        result_df = load_results_linkability(data_path)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    LinkRisk_Cancer[cancer] = ResultsLinkability

## Inference risk

In [ ]:
def load_results_inference(results): 
    
    with open(src_path, 'rb') as file:
        results_inference = pickle.load(file)
        
    ResultasDataset = {}
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    for tool, result_as_attributes in results_inference.items():
        tool_res = {}
        for feature in result_as_attributes:
            name_feature = feature[0]
            evaluator = feature[1]
    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[name_feature] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools

    return df

def extract_num_clincal(datapath, metadata, number_clinical):
    original_data = pd.read_csv(datapath, index_col = 0)
    clinical_features = original_data.columns.tolist()[0:number_clinical]
    numerical_type = ['numerical']
    numerical_features = [i.replace(".", "_") for i, value_type in metadata.items() if value_type in numerical_type and i in clinical_features]

    return numerical_features

def rearrange(data, numerical_clinical_features):
    df_numerical = data.loc[:,numerical_clinical_features]
    df_categorical = data.loc[:, ~df_results.columns.isin(numerical_clinical_features)]
    
    df_merged = pd.concat([df_numerical,df_categorical], axis = 1)

    return df_merged

In [ ]:
InferenceRisk_cancer = {}
for i, cancer in enumerate(cancers):
    Inference_as_seed = {}
    for seed in seeds:
        src_path = f'../{cancer}/Privacy/Inference/Seed_{seed}/results_inferences.pkl'
        # data_path = f'../{cancer}/Data/original_data.csv'
        df_results = load_results_inference(src_path)
        mean_risk = df_results.mean(axis = 1)
        Inference_as_seed[seed] = mean_risk
    InferenceRisk_cancer[cancer] = Inference_as_seed

## Score calculation

In [ ]:
OveralScore_dict = {}

for cancer in cancers:
    uni_df = pd.DataFrame(UniRisk_Cancer[cancer])
    multi_df = pd.DataFrame(MultiRisk_Cancer[cancer])
    link_df = pd.DataFrame(LinkRisk_Cancer[cancer])
    inference_df = pd.DataFrame(InferenceRisk_cancer[cancer])
    
    sdg_methods = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    overalscore_dict = {}
    
    for i, tool in enumerate(sdg_methods):
        # Trích xuất các dòng dữ liệu dưới dạng array
        uni_risk = uni_df.iloc[i, :].values
        multi_risk = multi_df.iloc[i, :].values
        link_risk = link_df.iloc[i, :].values
        inference_risk = inference_df.iloc[i, :].values
        
        # Gom các mảng lại thành một ma trận (stack) để tính toán theo trục
        # Mỗi cột trong 'combined' sẽ chứa 4 giá trị rủi ro tương ứng
        combined = np.array([uni_risk, multi_risk, link_risk, inference_risk])
        
        # Sử dụng np.nanmean để tính trung bình, bỏ qua các giá trị np.nan
        # axis=0 giúp tính trung bình theo từng cột (từng seed)
        overal_risk = np.nanmean(combined, axis=0)
        
        # Tính Privacy Score: 1 - rủi ro
        overal_score = 1 - overal_risk
        
        overalscore_dict[tool] = overal_score
    
    # Tạo DataFrame kết quả
    # Lưu ý: 'seeds' cần phải trùng khớp với số lượng phần tử trong overal_score
    overal_score_df = pd.DataFrame(overalscore_dict).T
    overal_score_df.columns = seeds  # Gán tên cột là các seeds
    
    # Lưu file
    overal_score_df.to_csv(f'overal_privacy_score_{cancer}.csv', index=True)
    OveralScore_dict[cancer] = overal_score_df

In [ ]:
for cancer, overal_score_df in OveralScore_dict.items():
    print(f'-----{cancer}-----')
    stat_df = overal_score_df.T.describe()
    for tool in stat_df.columns.tolist():
        print(f"{tool}: {stat_df.loc['mean', tool]} +/- {stat_df.loc['std', tool]}")
        

In [ ]:
score_df = OveralScore_dict['ccRCC']
data = score_df.T.to_dict()
result = {key: list(inner_dict.values()) for key, inner_dict in data.items()}

## **Bayesian estimation**

In [ ]:
Heatmap_Dict = {}
for cancer, score_df in OveralScore_dict.items():
    data = score_df.T.to_dict()
    result = {key: list(inner_dict.values()) for key, inner_dict in data.items()}

    Heatmap_Dict[cancer] = result

In [ ]:
from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

try:
    import baycomp
except ImportError as e:
    raise ImportError("baycomp is required. Install with: pip install baycomp") from e

from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap


# -----------------------------
# Nature-style + consistent colormap
# -----------------------------

METHOD_COLOR_SCHEME = {
    "palette_name": "Set2",
    # A default Set2-like palette defined explicitly as hex colors so it's stable across environments.
    "palette": [
        "#66c2a5",  # greenish
        "#fc8d62",  # orange
        "#8da0cb",  # blue
        "#e78ac3",  # pink
        "#a6d854",  # lime
        "#ffd92f",  # yellow
        "#e5c494",  # tan
        "#b3b3b3",  # grey
    ],
}

# Cancer-type colors (fixed identity across the manuscript)
# Chosen to be colorblind-friendly-ish and visually distinct.
CANCER_COLORS = {
    "ccRCC": "#4C72B0",     # blue
    "Melanoma": "#DD8452",  # orange
    "NSCLC": "#55A868",     # green
}

# Font preference for Nature-style figures (Helvetica if available, otherwise Arial/DejaVu Sans fallback)
# Reuse dataset colors from this module for consistency (methods/algorithms identity)
DATASET_COLORS = {
    "Avatars K5": "#66c2a5",         # greenish
    "Avatars K10": "#fc8d62",        # orange
    "CTGAN": "#8da0cb",              # blue
    "Gaussian Copula": "#e78ac3",    # pink
    "Synthpop": "#a6d854",           # lime
    "TVAE": "#ffd92f",               # yellow
}


NATURE_FONT = {
    "family": "sans-serif",
    "sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
}

STABILITY_PLOT_COLORS = {
    "grid": "#e6e6e6",
    "heatmap_bg": "#f7f7f7",
}

PBETTER_FOCUS_CMAP = LinearSegmentedColormap.from_list(
    "pbetter_focus",
    [
        (0.0, "#ffffff"),
        (0.5, "#ffffff"),
        (1.0, "#1a9850"),
    ],
)


def _set_nature_rcparams(fontsize: int = 11) -> None:
    """
    Apply Nature-like matplotlib rcParams (Helvetica/Arial and clean axes).

    Args:
        fontsize (int): Base font size for the figure.

    Raises:
        ValueError: If fontsize is not positive.
    """
    if fontsize <= 0:
        raise ValueError("fontsize must be a positive integer.")

    plt.rcParams.update(
        {
            "font.size": fontsize,
            "font.family": NATURE_FONT["family"],
            "font.sans-serif": NATURE_FONT["sans-serif"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.titlesize": fontsize + 1,
            "axes.labelsize": fontsize,
            "xtick.labelsize": fontsize - 1,
            "ytick.labelsize": fontsize - 1,
            "legend.fontsize": fontsize - 1,
        }
    )


def build_baycomp_comparison_df(
    method_to_scores: Dict[str, Sequence[float]],
    methods_order: Optional[Sequence[str]] = None,
    rope: float = 0.01,
) -> pd.DataFrame:
    """
    Compute pairwise Bayesian comparison probabilities (better/worse/equivalent)
    using baycomp.two_on_single for all ordered pairs of methods.

    Args:
        method_to_scores (Dict[str, Sequence[float]]): Mapping method -> list/array of scores across seeds.
        methods_order (Optional[Sequence[str]]): Optional method ordering. If None, uses dict insertion order.
        rope (float): ROPE threshold for practical equivalence.
        seed (int): Random seed passed to baycomp for reproducibility.

    Returns:
        pd.DataFrame: Comparison table with columns:
            ["Method 1", "Method 2", "Better Prob", "Worse Prob", "Equivalent Prob"].

    Raises:
        ValueError: If fewer than 2 methods are provided.
        ValueError: If any method has fewer than 2 finite scores.
        ValueError: If rope is not positive.
    """
    if rope <= 0:
        raise ValueError("rope must be > 0.")

    if not method_to_scores or len(method_to_scores) < 2:
        raise ValueError("method_to_scores must contain at least 2 methods.")

    if methods_order is None:
        methods_order = list(method_to_scores.keys())
    else:
        methods_order = list(methods_order)

    rows: List[dict] = []

    # Validate and coerce scores
    scores_np: Dict[str, np.ndarray] = {}
    for m in methods_order:
        if m not in method_to_scores:
            raise ValueError(f"Method '{m}' listed in methods_order but not found in method_to_scores.")
        x = np.asarray(method_to_scores[m], dtype=float)
        x = x[np.isfinite(x)]
        if x.size < 2:
            raise ValueError(f"Method '{m}' must have at least 2 finite scores for baycomp.")
        scores_np[m] = x

    # Compute ordered pair probabilities: P(m1 > m2), P(m1 < m2), P(|diff|<=rope)
    for i, m1 in enumerate(methods_order):
        for j, m2 in enumerate(methods_order):
            if i == j:
                continue
            probs = baycomp.two_on_single(scores_np[m1], scores_np[m2], rope=rope)
            better, worse, equiv = float(probs[0]), float(probs[2]), float(probs[1])

            rows.append(
                {
                    "Method 1": m1,
                    "Method 2": m2,
                    "Better Prob": better,
                    "Worse Prob": worse,
                    "Equivalent Prob": equiv,
                }
            )

    return pd.DataFrame(rows)


def plot_pbetter_heatmap_grid(
    cancer_to_method_scores: Dict[str, Dict[str, Sequence[float]]],
    cancers_order: Sequence[str] = ("ccRCC", "Melanoma", "NSCLC"),
    methods_order: Optional[Sequence[str]] = None,
    rope: float = 0.01,
    value_col: str = "Better Prob",
    figsize: Tuple[int, int] = (18, 5.5),
    annot: bool = True,
    fmt: str = ".2f",
    fontsize: int = 11,
    show: bool = True,
) -> Tuple[plt.Figure, np.ndarray, Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
    allowed = {"Better Prob", "Worse Prob", "Equivalent Prob"}
    if value_col not in allowed:
        raise ValueError(f"value_col must be one of {sorted(allowed)}.")

    # Resolve cancer order
    cancers_order = list(cancers_order)
    for c in cancers_order:
        if c not in cancer_to_method_scores:
            raise ValueError(f"Cancer '{c}' missing from cancer_to_method_scores.")

    # Resolve global method order
    if methods_order is None:
        union_methods: List[str] = []
        for c in cancers_order:
            for m in list(cancer_to_method_scores[c].keys()):
                if m not in union_methods:
                    union_methods.append(m)
        methods_order = union_methods
    else:
        methods_order = list(methods_order)

    _set_nature_rcparams(fontsize=fontsize)
    sns.set(style="white", rc={"axes.facecolor": STABILITY_PLOT_COLORS["heatmap_bg"]})

    fig, axes = plt.subplots(1, len(cancers_order), figsize=figsize)
    if len(cancers_order) == 1:
        axes = np.array([axes])

    norm = TwoSlopeNorm(vmin=0.0, vcenter=0.5, vmax=1.0)

    mats_by_cancer: Dict[str, pd.DataFrame] = {}
    comp_by_cancer: Dict[str, pd.DataFrame] = {}

    for ax, cancer in zip(axes, cancers_order):
        comp_df = build_baycomp_comparison_df(
            method_to_scores=cancer_to_method_scores[cancer],
            methods_order=methods_order,
            rope=rope,
        )
        comp_by_cancer[cancer] = comp_df

        mat = comp_df.pivot(index="Method 1", columns="Method 2", values=value_col)
        mat = mat.reindex(index=methods_order, columns=methods_order)

        # Set diagonal to 0.5 for visibility in heatmap, will overlay gray
        for m in methods_order:
            if m in mat.index and m in mat.columns:
                mat.loc[m, m] = 0.5

        mats_by_cancer[cancer] = mat

        sns.heatmap(
            mat,
            ax=ax,
            cmap=PBETTER_FOCUS_CMAP if value_col == "Better Prob" else "viridis",
            norm=norm if value_col == "Better Prob" else None,
            annot=annot,
            fmt=fmt,
            linewidths=0.5,
            linecolor="lightgray",
            cbar=(ax is axes[-1]),
            cbar_kws={"label": value_col} if (ax is axes[-1]) else None,
            square=True,
        )

        # Overlay gray rectangle on the diagonal
        for i, m in enumerate(methods_order):
            if m in mat.index and m in mat.columns:
                rect = plt.Rectangle(
                    (i, i), 1.0, 1.0,
                    facecolor='#D3D3D3',
                    edgecolor='lightgray',
                    linewidth=0.5,
                    zorder=11
                )
                ax.add_patch(rect)

        ax.set_title(cancer, fontsize=fontsize + 1, fontweight="bold", pad=10)
        cancer_color = CANCER_COLORS.get(cancer, "#333333")
        ax.plot(
            [0.02, 0.98],  # from left to right (axes coordinates)
            [1.02, 1.02],  # slightly above the axes; visually "under" the title
            transform=ax.transAxes,
            color=cancer_color,
            lw=4,
            clip_on=False,
        )
        ax.set_xlabel("Method 2", fontsize=fontsize)
        ax.set_ylabel("Method 1", fontsize=fontsize)
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    if show:
        plt.show()

    return fig, axes, mats_by_cancer, comp_by_cancer

In [ ]:
methods_order = ["Avatars K5", "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]

# 1) Comparison dataframe cho 1 cancer
cc_df_ccRCC = build_baycomp_comparison_df(
    method_to_scores=Heatmap_Dict["ccRCC"],
    methods_order=methods_order,
    rope=0.01,
)
cc_df_Melanoma = build_baycomp_comparison_df(
    method_to_scores=Heatmap_Dict["Melanoma"],
    methods_order=methods_order,
    rope=0.01,
)
cc_df_NSCLC = build_baycomp_comparison_df(
    method_to_scores=Heatmap_Dict["NSCLC"],
    methods_order=methods_order,
    rope=0.01,
)
# print(cc_df.head())

# 2) 3 heatmaps (ccRCC, Melanoma, NSCLC)
fig_d, axes, mats_by_cancer, comp_by_cancer = plot_pbetter_heatmap_grid(
    cancer_to_method_scores=Heatmap_Dict,
    cancers_order=["ccRCC", "Melanoma", "NSCLC"],
    methods_order=methods_order,
    rope=0.01,
    # seed=42,
    value_col="Better Prob",   # hoặc "Equivalent Prob"
    figsize=(18, 5),
    annot=True,
    fmt=".2f",
    fontsize=11,
)
fig_d.savefig("OverallPrivacy.png", dpi=300, bbox_inches="tight")
fig_d.savefig("OverallPrivacy.pdf", bbox_inches="tight", facecolor="white")
cc_df_ccRCC.to_csv('OverallPrivacy_ccRCC_Bayesian.csv')
cc_df_Melanoma.to_csv('OverallPrivacy_Melanoma_Bayesian.csv')
cc_df_NSCLC.to_csv('OverallPrivacy_NSCLC_Bayesian.csv')